In [3]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

### Название столбцов   ||   Описание
- **realSum**: Общая стоимость объявления на Airbnb. (*Числовое значение*)
- **room_type**: Тип предлагаемой комнаты (например, отдельная, общая и т. д.). (*Категориальное значение*)
- **room_shared**: Является ли комната общей. (*Булево значение*)
- **room_private**: Является ли комната частной или нет. (*Булево значение*)
- **person_capacity**: Максимальное количество человек, которые могут разместиться в комнате. (*Числовое значение*)
- **host_is_superhost**: Является ли хозяин суперхозяином или нет. (*Булево значение*)
- **multi**: Предлагается ли в объявлении несколько комнат или нет. (*Булево значение*)
- **biz**: Указывает, предназначено ли объявление для деловых целей. (*Булево значение*)
- **cleanliness_rating**: Рейтинг чистоты объявления. (*Числовое значение*)
- **guest_satisfaction_overall**: Общий рейтинг удовлетворенности гостей объявлением. (*Числовое значение*)
- **bedrooms**: Количество спален в объявлении. (*Числовое значение*)
- **dist**: Расстояние от центра города. (*Числовое значение*)
- **metro_dist**: Расстояние до ближайшей станции метро. (*Числовое значение*)
- **attr_index**: Индекс привлекательности места, указанного в списке (*Числовое значение*)
- **attr_index_norm**: Нормализованный индекс привлекательности (0–100) (*Числовое значение*)
- **rest_index**: Индекс ресторанов места, указанного в списке (*Числовое значение*)
- **rest_index_norm**: Нормализованный индекс ресторанов (0–100) (*Числовое значение*)
- **lng**: Долгота объекта. (*Числовое значение*)
- **lat**: Широта объекта. (*Числовое значение*)

#### **Цель:**

Моя главная цель **создать универсальный ETL** для данных используя объект **class**. Этот ETL будет DataPreprocessing class.

* **init**: Принимает навчальные параметры | **rows**: *pd.DataFrame*, **null_threshold**: *float*, **drop_features**: *list*, **save_path**: *str*
* **get_new_df()**: Возвращяет объект **df**: *pd.DataFrame*
* **get()**: Возвращяет объект **rows_numpy_arr** по одному через **yield**
* **change_numpy_arr(x)**: Конвертирует **x** в *np.array* и перезаписывает объект **rows_numpy_arr**
* **get_features_type()**: Возвращяет список *типов данных*, *количество* колонок в **df** с этим типом данных и *наименование* колонок 
* **drop_dublicates**(): Удаляет дубликаты в объекте **df**
* **change_features_types(change_features_dict)**: Меняет по списку *change_features_dict* типы данных в объекте **df**
* **drop_specified_features()**: Удаляет список колонок которых я указал как параметр *drop_features* из объекта *df*
* **null_features()**: Дает соотношение пустых данных по колонкам из объекта *df* и удаляет те колонки у которых это значение меньше параметра *null_threshold*
* **get_log_scaler(x)**: *np.log1p* трансформация для данных *x* которые имеют "хвост"
* **get_absolute_max_scaler(x)**: Делит данные *x* на обсолютное максимальное значение, это я использовал для колонок которые связанные с оценкой
* **preprocessing_int_float_features(num_features_methods_dict)**: Делает предобработку числовым данным (*MinMaxScaler*, *LogScaler*, *AbsoluteMaxScaler*) | Передаем список **num_features_methods_dict** с наименованием колонок и методами которые мы будем использовать для них 
* **preprocessing_cat_features(cat_features_methods_dict)**: Делает предобработку категориальных данных (*OneHotEncoder*, *LabelEncoder*) | Передаем список **cat_features_methods_dict** с наименованием колонок и методами которые мы будем использовать для них
* **change_columns_name()**: Просто заменяет [' ', '/'] на '_'
* **save()**: Сохраняет объект *df* с наименование и путём которые мы передаем в параметр *save_path*

In [4]:
folder_path = Path(r"..\data\original")
files_list = list(folder_path.glob('*.csv'))

In [5]:
class DataPreprocessing:
    
    def __init__(self, rows: pd.DataFrame, null_threshold: float, drop_features: list, save_path: str) -> None:
        self.df = rows
        self.null_threshold = null_threshold
        self.drop_features = drop_features
        self.save_path = save_path
        
        self.rows_numpy_arr = np.array(rows)
        
    def get(self):
        
        for i in self.rows_numpy_arr:
            yield i
    
    def get_new_df(self):
        
        return self.df
        
    def change_numpy_arr(self, x: pd.DataFrame) -> np.array:
        # Конвертирует pd.DataFrame -> np.array
        # Этот метод будет полезен, когда мне нужно будет работать с данными по строчно
        
        self.rows_numpy_arr = np.array(x)
        
        return self.rows_numpy_arr
            
    def get_features_type(self) -> dict:
        # Возвращяет {'type': [count, [features]]}
        
        types = ['int', 'float', 'object', 'bool']
        types_count_dict = dict()
        
        for t in types:
            types_count_dict[t] = [len(self.df.select_dtypes(include=t).columns), list(self.df.select_dtypes(include=t).columns)]
            
        print("Return Features Types -------------------- ✅")
        
        return types_count_dict

    def drop_dublicates(self) -> None:
        # Удаляет дубликаты
    
        count_duplicates = len(self.df[self.df.duplicated()])
        print(f"Количество дубликатов: {count_duplicates}")
        
        self.df = self.df.drop_duplicates()
        
        print("Drop Duplicates -------------------- ✅")
        
        
    def change_features_types(self, change_features_dict: dict, save_changes = False) -> pd.DataFrame:
        # Я вручную указываю для каких колонок поменять тип
        
        df_new = self.df.copy()
        
        for features, new_type in change_features_dict.items():
            df_new[features] = df_new[features].astype(new_type)
        
        if save_changes == True:
            self.df = df_new
            
        print("Change Features Types -------------------- ✅")
        
        return df_new
        
    
    def drop_specified_features(self) -> None:
        # Удаляет колонки которых я посчитал не нужными
        
        self.df = self.df.drop(columns=self.drop_features)
        
        print(f"Drop Specified Features | {self.drop_features} -------------------- ✅")
    
    
    def null_features(self) -> None:
        # Дает информацию про пустотам в данных и удаляет колонки, которых соотношение пустот больше лимита
        
        nulls_features = self.df.isna().mean()
        
        print(f"Ratio of nulls in columns: \n{nulls_features}")
        
        drop_null_cols = nulls_features[nulls_features > self.null_threshold].index.to_list()
        self.df = self.df.drop(columns=drop_null_cols)
        
        print(f"Droped columns with nulls | {drop_null_cols}")
        
        print("Get And Drop Null Features -------------------- ✅")
    
    
    def get_log_scaler(self, x):
        # Используется для данных имеющий хвост
        return np.log1p(x)

    
    def get_absolute_max_scaler(self, x):
        # Делит данные на обсолютное максимальное значение
        max_abs = np.max(np.abs(x))
        
        return x/max_abs
    
    
    def preprocessing_int_float_features(self, num_features_methods_dict: dict) -> pd.DataFrame:
        # Предобработка числовых данных
        
        for method, features in num_features_methods_dict.items():
            if method == 'MinMaxScaler':
                ss = MinMaxScaler()
                self.df[features] = ss.fit_transform(
                    self.df[features]
                )
            if method == 'LogScaler':
                for f in features:
                    self.df[f] = self.get_log_scaler(self.df[f])
            if method == 'AbsoluteMaxScaler':
                for f in features:
                    self.df[f] = self.get_absolute_max_scaler(self.df[f])
        
        print("Preprocessing Int Float Features -------------------- ✅")
        
        return self.df
    
    
    def preprocessing_cat_features(self, cat_features_methods_dict: dict) -> pd.DataFrame:
        # Предобработка категориальные данных
        
        for method, features in cat_features_methods_dict.items():
            if method == 'OneHotEncoder':
                ohe = OneHotEncoder(handle_unknown='ignore',
                                    sparse_output=False)
                
                encoded = ohe.fit_transform(
                    self.df[features]
                )
                
                encoded_df = pd.DataFrame(encoded, 
                                          columns=ohe.get_feature_names_out(features), 
                                          index=self.df.index
                                        )

                self.df = pd.concat(
                    [self.df.drop(columns=features), encoded_df],
                    axis=1
                )
            if method == 'LabelEncoder':
                for feature in features:
                    le = LabelEncoder()

                    self.df[feature] = le.fit_transform(
                        self.df[feature].astype(str)
                    )
        
        print("Preprocessing Categorical Features -------------------- ✅")      
            
        return self.df
    
    
    def change_columns_name(self) -> None:
        # Просто заменяет [' ', '/'] на '_'
        
        self.df.columns = self.df.columns.str.replace(' ', '_')
        self.df.columns = self.df.columns.str.replace('/', '_')
        
        print("Change Columns Name (' ', '/') -------------------- ✅")
        
    
    def save(self) -> None:
        self.df.to_csv(self.save_path)
 

In [6]:
drop_features = ['lng', 'lat', 'attr_index_norm', 'rest_index_norm', 'room_shared', 'room_private']
num_limit = 0.4
df_final = pd.DataFrame()

for i in range(len(files_list)):
    file_path = files_list[i]
    file_name = file_path.name
    
    save_path = '..\\data\\preprocessed\\' + file_name

    df = pd.read_csv(file_path, index_col=0)
    
    time_str = file_name.split('_')
    city_name = time_str[0]
    day = (time_str[1]).split('.')[0]
    
    df['city_name'] = city_name
    df['day'] = day
    
    me = DataPreprocessing(df, num_limit, drop_features, save_path)
    
    # Pipline
    me.drop_specified_features()
    me.drop_dublicates()
    
    chng_types = {
        'person_capacity': 'int',
        'multi': 'bool',
        'biz': 'bool'
    }
    me.change_features_types(change_features_dict=chng_types, save_changes=True)
    
    me.null_features()
    
    cat_features_methods_dict = {
        'OneHotEncoder': ['room_type']
    }

    me.preprocessing_cat_features(cat_features_methods_dict)
    me.change_columns_name()
    
    chng_types = {
        'room_type_Entire_home_apt': 'bool',
        'room_type_Private_room': 'bool',
        'room_type_Shared_room': 'bool'
    }
    me.change_features_types(change_features_dict=chng_types, save_changes=True)
    
    df_new = me.get_new_df()
    
    df_new.head()
        
    if i == 0:
        df_final = df_new.copy()
    else:
        df_final = pd.concat([df_final, df_new], axis=0)
        
df_final.to_csv("..\\data\\preprocessed\\final_data.csv")

Drop Specified Features | ['lng', 'lat', 'attr_index_norm', 'rest_index_norm', 'room_shared', 'room_private'] -------------------- ✅
Количество дубликатов: 0
Drop Duplicates -------------------- ✅
Change Features Types -------------------- ✅
Ratio of nulls in columns: 
realSum                       0.0
room_type                     0.0
person_capacity               0.0
host_is_superhost             0.0
multi                         0.0
biz                           0.0
cleanliness_rating            0.0
guest_satisfaction_overall    0.0
bedrooms                      0.0
dist                          0.0
metro_dist                    0.0
attr_index                    0.0
rest_index                    0.0
city_name                     0.0
day                           0.0
dtype: float64
Droped columns with nulls | []
Get And Drop Null Features -------------------- ✅
Preprocessing Categorical Features -------------------- ✅
Change Columns Name (' ', '/') -------------------- ✅
Change Featu

In [8]:
save_path = '..\\data\\preprocessed\\final_data.csv'
me = DataPreprocessing(df_final, num_limit, drop_features, save_path)

num_features_methods_dict = {
    'StandardScaler': ['dist', 'metro_dist', 'attr_index', 'rest_index'],
    'LogScaler': ['realSum'],
    'AbsoluteMaxScaler': ['cleanliness_rating', 'guest_satisfaction_overall']
}
me.preprocessing_int_float_features(num_features_methods_dict)

cat_features_methods_dict = {
    'OneHotEncoder': ['day'],
    'LabelEncoder': ['city_name']
}
me.preprocessing_cat_features(cat_features_methods_dict)
me.change_columns_name()

chng_types = {
    'day_weekdays': 'bool',
    'day_weekends': 'bool'
}
me.change_features_types(change_features_dict=chng_types, save_changes=True)
me.save()

Preprocessing Int Float Features -------------------- ✅
Preprocessing Categorical Features -------------------- ✅
Change Columns Name (' ', '/') -------------------- ✅
Change Features Types -------------------- ✅
